In [1]:
import numpy as np
import polars as pl
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_validate, cross_val_predict, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, roc_auc_score, make_scorer

import seaborn as sn
import matplotlib.pyplot as plt

In [2]:
# read tsv clinical matrix
clinicalMatrix = pd.read_csv("brca_data/TCGA.BRCA.sampleMap-BRCA_clinicalMatrix.tsv", sep="\t")

In [2]:
file_path = 'MOGLAM/BRCA_split/BRCA/1_featname.csv'  # Replace 'your_file.txt' with the actual file path

# Open the file and read lines into a list
with open(file_path, 'r') as file:
    file_content = file.readlines()

# Optionally, strip newline characters from each line
gene_list = [line.strip().split('|')[0] for line in file_content][1:]

# Now, file_content is a list where each element corresponds to a line in the file
print(gene_list)

['A2ML1', 'ABAT', 'ABCA13', 'ABCC11', 'ABCC8', 'ABCG1', 'ABCG2', 'ABLIM3', 'ABTB2', 'ACADSB', 'ACE2', 'ACOT4', 'ACOX2', 'ACTL6A', 'ACTR3B', 'ACVR1B', 'ADAMTS15', 'ADCY9', 'ADD1', 'AFF3', 'AGR2', 'AGR3', 'AKR1E2', 'AKR7A3', 'ALAD', 'AMD1', 'AMY1A', 'ANKLE1', 'ANKMY1', 'ANKRA2', 'ANKRD30A', 'ANKRD30B', 'ANKRD42', 'ANKRD45', 'ANKS6', 'ANLN', 'ANP32E', 'ANXA8L2', 'ANXA9', 'APBB2', 'APH1B', 'APPL2', 'ARHGAP11A', 'ARHGEF4', 'ARL9', 'ARSG', 'ART3', 'AR', 'ASNS', 'ASTN2', 'ATAD2', 'ATL2', 'ATOH7', 'ATP1B3', 'ATP5C1', 'ATP6V1C2', 'ATP7B', 'ATP8B1', 'AURKA', 'AURKB', 'B3GNT4', 'B3GNT5', 'BAIAP2L2', 'BBOX1', 'BBS1', 'BBS4', 'BBS5', 'BCAM', 'BCAS1', 'BCL11A', 'BCL2', 'BECN1', 'BHLHE40', 'BIRC5', 'BLM', 'BPI', 'BRIX1', 'BTG2', 'BTG3', 'BUB1', 'BYSL', 'C10orf116', 'C10orf26', 'C10orf32', 'C10orf90', 'C11orf75', 'C11orf82', 'C11orf86', 'C12orf11', 'C12orf48', 'C12orf72', 'C13orf27', 'C14orf174', 'C14orf45', 'C15orf23', 'C15orf42', 'C16orf45', 'C16orf57', 'C16orf61', 'C16orf71', 'C17orf28', 'C19orf51'

In [4]:
"""
Fetch interactions for a specific gene or gene list
"""

import requests
import json

request_url = "https://webservice.thebiogrid.org/interactions/"
# List of genes to search for
# geneList = ["STE11", "NMD4"]  # Yeast Genes STE11 and NMD4
evidenceList = ["POSITIVE GENETIC", "PHENOTYPIC ENHANCEMENT"]

# These parameters can be modified to match any search criteria following
# the rules outlined in the Wiki: https://wiki.thebiogrid.org/doku.php/biogridrest
params = {
    "accesskey": "7bcc32a723cda4c96a00c69790c6c26e",
    "format": "tab3",  # Return results in TAB2 format
    "geneList": "|".join(gene_list[0]),  # Must be | separated
    "searchNames": "true",  # Search against official names
    "includeInteractors": "true",  # Set to true to get any interaction involving EITHER gene, set to false to get interactions between genes
    # "taxId": 559292,  # Limit to Saccharomyces cerevisiae
    "evidenceList": "|".join(evidenceList),  # Exclude these two evidence types
    "includeEvidence": "false",  # If false "evidenceList" is evidence to exclude, if true "evidenceList" is evidence to show
    "includeHeader": "true",
}

# Additional options to try, you can uncomment them as necessary
# params["start"] = 5 # Specify where to start fetching results from if > 10,000 results being returned
# params["max"] = 10 # Specify the number of results to return, max is 10,000
# params["interSpeciesExcluded"] = "false" # true or false, If ‘true’, interactions with interactors from different species will be excluded (ex. no Human -> Mouse interactions)
# params["selfInteractionsExcluded"] = "false" # true or false, If ‘true’, interactions with one interactor will be excluded. (ex. no STE11 -> STE11 interactions)
# params["searchIds"] = "false" # true or false, If ‘true’, ENTREZ_GENE, ORDERED LOCUS and SYSTEMATIC_NAME (orf) will be examined for a match with the geneList
# params["searchSynonyms"] = "false" # true or false, If ‘true’, SYNONYMS will be examined for a match with the geneList
# params["searchBiogridIds"] = "false" # true or false, If ‘true’, BIOGRID INTERNAL IDS will be examined for a match with the geneList
# params["excludeGenes"] = "false" # true or false, If 'true' the geneList becomes a list of genes to EXCLUDE rather than to INCLUDE
# params["includeInteractorInteractions"] = "true" # true or false, If ‘true’ interactions between the geneList’s first order interactors will be included. Ignored if includeInteractors is ‘false’ or if excludeGenes is set to ‘true’.
# params["htpThreshold"] = 50 # Any publication with more than this many interactions will be excluded
# params["throughputTag"] = "any" # any, low, high. If set to low, only `low throughput` interactions will be returned, if set to high, only `high throughput` interactions will be returned
# params["additionalIdentifierTypes"] = "SGD|FLYBASE|REFSEQ" # You can specify a | separated list of additional identifier types to search against (see get_identifier_types.py)

r = requests.get(request_url, params=params)
interactions = r.text

# Pretty print out the results
print(len(interactions))

1707283


In [ ]:
# write interactions to file
with open('test_interactions.tsv', 'w') as f:
    f.write(interactions)

In [8]:
# print(interactions)
# int_df = pl.read_csv(interactions, separator='\t')

ComputeError: invalid glob pattern given

# Class counts

In [16]:
# get two columns, drop nan and reindex from 0
cM = clinicalMatrix[['sampleID', 'PAM50_mRNA_nature2012']].dropna().reset_index(drop=True)
# print(cM)

# filter rows with PAM50_mRNA_nature2012 == 'Normal'
cM = cM[cM['PAM50_mRNA_nature2012'] != 'Normal-like'].reset_index(drop=True)

names, counts = np.unique(cM['PAM50_mRNA_nature2012'].to_numpy(), return_counts=True)
for i, j in zip(names, counts):
    print(f"{i}: {j}")

Basal-like: 98
HER2-enriched: 58
Luminal A: 231
Luminal B: 127


In [29]:
# get two columns, drop nan and reindex from 0
cM = clinicalMatrix[['sampleID', 'PAM50Call_RNAseq']].dropna().reset_index(drop=True)
print(cM)
names, counts = np.unique(cM['PAM50Call_RNAseq'].to_numpy(), return_counts=True)
for i, j in zip(names, counts):
    print(f"{i}: {j}")

            sampleID PAM50Call_RNAseq
0    TCGA-A1-A0SB-01           Normal
1    TCGA-A1-A0SD-01             LumA
2    TCGA-A1-A0SE-01             LumA
3    TCGA-A1-A0SF-01             LumA
4    TCGA-A1-A0SG-01             LumA
..               ...              ...
951  TCGA-GM-A2DM-01             LumA
952  TCGA-GM-A2DN-01             LumA
953  TCGA-GM-A2DO-01             LumB
954  TCGA-GM-A3NY-01             LumA
955  TCGA-HN-A2NL-01            Basal

[956 rows x 2 columns]
Basal: 142
Her2: 67
LumA: 434
LumB: 194
Normal: 119
